In [1]:
import pandas as pd
import inspect

from business_entity_resolution.components.data_ingestion import DataIngestion
from business_entity_resolution.components.data_transformation import DataTransformation
from business_entity_resolution.components.candidate_generation import CandidateGeneration


In [2]:
import os

PROJECT_ROOT = r"C:\Users\kaila\OneDrive\Desktop\Amazon ML"

os.chdir(PROJECT_ROOT)

print("Working directory:")
print(os.getcwd())

Working directory:
C:\Users\kaila\OneDrive\Desktop\Amazon ML


In [3]:
ingestion = DataIngestion()

source1, source2, source3, ground_truth = ingestion.initiate_data_ingestion()

print("Source 1:", source1.shape)
print("Source 2:", source2.shape)
print("Source 3:", source3.shape)
print("Ground truth:", ground_truth.shape)


Source 1: (2206821, 4)
Source 2: (5034616, 4)
Source 3: (5285603, 4)
Ground truth: (2206821, 2)


In [4]:
print(inspect.getsource(CandidateGeneration.create_blocking_keys))


    @staticmethod
    def create_blocking_keys(df):

        df = df.copy()

        # -----------------------------
        # Name-based blocking
        # -----------------------------

        df["name_block"] = (
            df["business_name_clean"]
            .fillna("")
            .astype(str)
            .str[:4]
        )

        df["name_blocking_key"] = (
            df["country"]
            .fillna("")
            .astype(str)
            .str.lower()
            + "_"
            + df["name_block"]
        )

        # -----------------------------
        # Address-based blocking
        # -----------------------------

        df["address_block"] = (
            df["business_address_clean"]
            .fillna("")
            .astype(str)
            .str[:6]
        )

        df["address_blocking_key"] = (
            df["country"]
            .fillna("")
            .astype(str)
            .str.lower()
            + "_"
            + df["address_block"]
        )

In [5]:
matched_s1_ids = (
    ground_truth[
        ground_truth["matched_entity_ids"].notna()
        & ground_truth["matched_entity_ids"].ne("")
    ]["source1_entity_id"]
    .drop_duplicates()
)

selected_s1_ids = (
    matched_s1_ids
    .sample(n=100, random_state=42)
    .tolist()
)

small_source1 = source1[
    source1["entity_id"].isin(selected_s1_ids)
].copy()

small_ground_truth = ground_truth[
    ground_truth["source1_entity_id"].isin(selected_s1_ids)
].copy()

print("Selected S1:", len(selected_s1_ids))


Selected S1: 100


In [6]:
def get_matching_ids(ground_truth_df, source_prefix):
    matched_ids = (
        ground_truth_df["matched_entity_ids"]
        .dropna()
        .astype(str)
        .str.split(",")
        .explode()
        .str.strip()
    )

    return matched_ids[
        matched_ids.str.startswith(source_prefix)
    ].unique()


true_s2_ids = get_matching_ids(
    small_ground_truth,
    "S2-"
)

true_s3_ids = get_matching_ids(
    small_ground_truth,
    "S3-"
)

small_source2_true = source2[
    source2["entity_id"].isin(true_s2_ids)
].copy()

small_source3_true = source3[
    source3["entity_id"].isin(true_s3_ids)
].copy()

print("True S2 records:", len(true_s2_ids))
print("True S3 records:", len(true_s3_ids))


True S2 records: 195
True S3 records: 179


In [7]:
#add non matching records
extra_s2 = source2[
    ~source2["entity_id"].isin(true_s2_ids)
].head(max(5000 - len(small_source2_true), 0))

extra_s3 = source3[
    ~source3["entity_id"].isin(true_s3_ids)
].head(max(5000 - len(small_source3_true), 0))

small_source2 = pd.concat(
    [small_source2_true, extra_s2],
    ignore_index=True
)

small_source3 = pd.concat(
    [small_source3_true, extra_s3],
    ignore_index=True
)

print("S1 sample:", small_source1.shape)
print("S2 sample:", small_source2.shape)
print("S3 sample:", small_source3.shape)


S1 sample: (100, 4)
S2 sample: (5000, 4)
S3 sample: (5000, 4)


In [8]:
transformer = DataTransformation()

small_source1 = transformer.transform_source(small_source1)
small_source2 = transformer.transform_source(small_source2)
small_source3 = transformer.transform_source(small_source3)

print("Transformation complete.")


Transformation complete.


In [9]:
#genarte candidates
candidate_generator = CandidateGeneration()

candidates_s2, candidates_s3 = (
    candidate_generator.generate_candidates(
        small_source1,
        small_source2,
        small_source3
    )
)

print("S2 candidate pairs:", len(candidates_s2))
print("S3 candidate pairs:", len(candidates_s3))


S2 candidate pairs: 1032
S3 candidate pairs: 1180


In [10]:
#get true pairs
ground_truth_pairs = (
    small_ground_truth[
        ["source1_entity_id", "matched_entity_ids"]
    ]
    .dropna(subset=["matched_entity_ids"])
    .copy()
)

ground_truth_pairs["matched_entity_id"] = (
    ground_truth_pairs["matched_entity_ids"]
    .str.split(",")
)

ground_truth_pairs = (
    ground_truth_pairs
    .explode("matched_entity_id")
)

ground_truth_pairs["matched_entity_id"] = (
    ground_truth_pairs["matched_entity_id"]
    .str.strip()
)

ground_truth_pairs = ground_truth_pairs[
    ground_truth_pairs["matched_entity_id"].ne("")
].drop_duplicates()

true_s2_pairs = ground_truth_pairs[
    ground_truth_pairs["matched_entity_id"].str.startswith("S2-")
].copy()

true_s3_pairs = ground_truth_pairs[
    ground_truth_pairs["matched_entity_id"].str.startswith("S3-")
].copy()

print("Total true pairs:", len(ground_truth_pairs))
print("True S2 pairs:", len(true_s2_pairs))
print("True S3 pairs:", len(true_s3_pairs))


Total true pairs: 374
True S2 pairs: 195
True S3 pairs: 179


In [11]:
def calculate_blocking_recall(
    true_pairs,
    candidate_pairs,
    candidate_id_column
):
    true_set = set(
        zip(
            true_pairs["source1_entity_id"],
            true_pairs["matched_entity_id"]
        )
    )

    candidate_set = set(
        zip(
            candidate_pairs["entity_id_s1"],
            candidate_pairs[candidate_id_column]
        )
    )

    found_pairs = true_set.intersection(candidate_set)

    return {
        "true_pairs": len(true_set),
        "candidate_pairs": len(candidate_set),
        "true_pairs_found": len(found_pairs),
        "true_pairs_missed": len(true_set - candidate_set),
        "blocking_recall": (
            len(found_pairs) / len(true_set)
            if true_set else 0.0
        )
    }


s2_result = calculate_blocking_recall(
    true_s2_pairs,
    candidates_s2,
    "entity_id_s2"
)

s3_result = calculate_blocking_recall(
    true_s3_pairs,
    candidates_s3,
    "entity_id_s3"
)

print("====================================")
print("BLOCKING RECALL RESULTS")
print("====================================")
print(f"S2 Blocking Recall: {s2_result['blocking_recall']:.4f}")
print(f"S3 Blocking Recall: {s3_result['blocking_recall']:.4f}")
print(
    f"S2 True Pairs Found: "
    f"{s2_result['true_pairs_found']} / {s2_result['true_pairs']}"
)
print(
    f"S3 True Pairs Found: "
    f"{s3_result['true_pairs_found']} / {s3_result['true_pairs']}"
)


BLOCKING RECALL RESULTS
S2 Blocking Recall: 0.9846
S3 Blocking Recall: 0.9497
S2 True Pairs Found: 192 / 195
S3 True Pairs Found: 170 / 179


In [ ]:
s2_true_set = set(
    zip(
        true_s2_pairs["source1_entity_id"],
        true_s2_pairs["matched_entity_id"]
    )
)

s2_candidate_set = set(
    zip(
        candidates_s2["entity_id_s1"],
        candidates_s2["entity_id_s2"]
    )
)

missed_s2_pairs = s2_true_set - s2_candidate_set

missed_s2_df = pd.DataFrame(
    list(missed_s2_pairs),
    columns=[
        "source1_entity_id",
        "matched_entity_id"
    ]
)

print("Missed S2 pairs:", len(missed_s2_df))
missed_s2_df.head(20)


Missed S2 pairs: 3


,source1_entity_id,matched_entity_id
0,S1-571004885,S2-434751720
1,S1-134755367,S2-214030604
2,S1-491000076,S2-193933459


In [17]:
missed_s2_details = (
    missed_s2_df
    .merge(
        small_source1[
            [
                "entity_id",
                "business_name",
                "business_address",
                "country"
            ]
        ],
        left_on="source1_entity_id",
        right_on="entity_id",
        how="left"
    )
    .drop(columns=["entity_id"])
    .merge(
        small_source2[
            [
                "entity_id",
                "business_name",
                "business_address",
                "country"
            ]
        ],
        left_on="matched_entity_id",
        right_on="entity_id",
        how="left",
        suffixes=("_s1", "_s2")
    )
    .drop(columns=["entity_id"])
)

missed_s2_details.head(20)


,source1_entity_id,matched_entity_id,business_name_s1,business_address_s1,country_s1,business_name_s2,business_address_s2,country_s2
0,S1-571004885,S2-434751720,Gujarat Enterprises Private Limited,"Vardhman Plaza Lsc Sector 6, Dwarka, Plot No. ...",India,गुजरात एंटरप्राइजेज प्राइवेट लिमिटेड,"PLOT NO. 8 F/F, NEW DELHI, SOUTH WEST DELHI, D...",India
1,S1-134755367,S2-214030604,Om Consultancy Pvt Ltd,"23 G Floor, World Trade Centre, Barakhamba Ave...",India,ओम कंसल्टेंसी प्रा. लि.,"G FLOOR, NEW DELHI, Delhi",India
2,S1-491000076,S2-193933459,Minerva Trading Private Limited,"Plot No.49, Commercial Mandi Near Cat Square, ...",India,Mr Minerva Private Limited Service,"H.NO #49 , COMMERCIAL MANDI NEAR CAT SQUARE, R...",India


In [19]:
display(missed_s2_details)

# -----------------------------
# Find missed S3 pairs
# -----------------------------

s3_true_set = set(
    zip(
        true_s3_pairs["source1_entity_id"],
        true_s3_pairs["matched_entity_id"]
    )
)

s3_candidate_set = set(
    zip(
        candidates_s3["entity_id_s1"],
        candidates_s3["entity_id_s3"]
    )
)

missed_s3_pairs = s3_true_set - s3_candidate_set

missed_s3_df = pd.DataFrame(
    list(missed_s3_pairs),
    columns=[
        "source1_entity_id",
        "matched_entity_id"
    ]
)

print("Missed S3 pairs:", len(missed_s3_df))

missed_s3_details = (
    missed_s3_df
    .merge(
        small_source1[
            [
                "entity_id",
                "business_name",
                "business_address",
                "country"
            ]
        ],
        left_on="source1_entity_id",
        right_on="entity_id",
        how="left"
    )
    .drop(columns=["entity_id"])
    .merge(
        small_source3[
            [
                "entity_id",
                "business_name",
                "business_address",
                "country"
            ]
        ],
        left_on="matched_entity_id",
        right_on="entity_id",
        how="left",
        suffixes=("_s1", "_s3")
    )
    .drop(columns=["entity_id"])
)

display(missed_s3_details)

,source1_entity_id,matched_entity_id,business_name_s1,business_address_s1,country_s1,business_name_s2,business_address_s2,country_s2
0,S1-571004885,S2-434751720,Gujarat Enterprises Private Limited,"Vardhman Plaza Lsc Sector 6, Dwarka, Plot No. ...",India,गुजरात एंटरप्राइजेज प्राइवेट लिमिटेड,"PLOT NO. 8 F/F, NEW DELHI, SOUTH WEST DELHI, D...",India
1,S1-134755367,S2-214030604,Om Consultancy Pvt Ltd,"23 G Floor, World Trade Centre, Barakhamba Ave...",India,ओम कंसल्टेंसी प्रा. लि.,"G FLOOR, NEW DELHI, Delhi",India
2,S1-491000076,S2-193933459,Minerva Trading Private Limited,"Plot No.49, Commercial Mandi Near Cat Square, ...",India,Mr Minerva Private Limited Service,"H.NO #49 , COMMERCIAL MANDI NEAR CAT SQUARE, R...",India


Missed S3 pairs: 9


,source1_entity_id,matched_entity_id,business_name_s1,business_address_s1,country_s1,business_name_s3,business_address_s3,country_s3
0,S1-571004885,S3-950784712,Gujarat Enterprises Private Limited,"Vardhman Plaza Lsc Sector 6, Dwarka, Plot No. ...",India,गुजरात एंटरप्राइजेज प्राइवेट लिमिटेड,"Plot No. #8 F/f, South West Delhi, Dwarka New ...",India
1,S1-89282797,S3-642502568,Timber Inc,"11504 1/2 Spruce Canyon Drive, Austin, TX",US,Ínc Timber,"1150414 Spruce Canyon Dr, Austin, Texas",US
2,S1-93926066,S3-156364994,Paras Energy,"C/O P.Manikantha, Po-Charamula, Telengashi, Na...",India,PE,"Nabarangpur, C/O P.Manikantha, Po-Charamula, T...",India
3,S1-83174673,S3-755114994,Supreme It Private Limited,"Office No S 07 82Haware, Centurion Plt88-91 Se...",India,सुप्रीम आईटी प्राइवेट लिमिटेड,"Offiec No S ##07 82Haware, Centurion Plt88-91 ...",India
4,S1-449306864,S3-535057563,A Cure 2 All PLLC,"OK, 14840 60th Avenue E, Collinsville",US,A All 2 Cure PLLC,"60th Ave East, Collinsville Township, Oklahoma",US
5,S1-190133982,S3-716445195,MS Consultancy Corp,Shymala Appts 1St Floor Flat No. 4 Opp Ratna H...,India,M5 Consultancy Corp,"Pune, MH, Pune, Shymala Appts 1St Floor Flat N...",India
6,S1-20062520,S3-612765588,Magruder Prairie Services Group,"TX, Magnolia, 30124 Nichols Sawmill Road",US,The Magruder Prairie Services Group,"30122 Nichols Sawmill Rd, Magnolia, Texas",US
7,S1-571004885,S3-714238060,Gujarat Enterprises Private Limited,"Vardhman Plaza Lsc Sector 6, Dwarka, Plot No. ...",India,गुजरात एंटरप्राइजेज प्राइवेट लिमिटेड,"Plot No. #8 F/f, Vardhman Plaza Lsc Sector 6, ...",India
8,S1-387663358,S3-580474124,Sauhard Welfare Society Limited,"Shop No.26 & 27, Ground Floor, Parsavnath Metr...",India,M/s Sauhard Society Welfare Limited,"North West, DL, Shop No.2-6 & 27, New Delhi",India
